# 6 · The final model, trained on the stream

The model `4_upscale_strategies.ipynb` picked: the **reparameterized 32-16**. While
training, each convolution is three parallel branches; afterwards they are merged into
one, so it ships as the plain 32-16 graph from notebook 3. The loss matches the score:
L1 with brightness weighted 6:1:1.

Notebook 4 compared models on drawn desktops with a clean downscale. This one trains on
the real data from `0_data_preprocess.ipynb` - photos and YouTube video sent through the
stream's H.264 - and publishes the result at the tile size from notebook 5 (320x180).

Steps: data -> model -> train -> results by bitrate and content -> merge -> save -> export.

Needs: `0_data_preprocess.ipynb`. Results: `summary.md`.

In [ ]:
"""Settings and data."""

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Subset

from helpers import degrade, metrics, models, patches, sources, training, webexport
from helpers.training import DATA_DIR, DEVICE

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_reparam.pt"

WIDTHS = (32, 16)
STEM = 5
EXPAND = 4           # how much wider the extra training branch is
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0

STEP_WIDTH, STEP_HEIGHT = 320, 180       # LR pixels one tile adds to the picture (notebook 5)
FRAME_WIDTH, FRAME_HEIGHT = 1920, 1080   # the whole-frame graph
NAME = "reparam_32-16"

training.seed_everything(SEED)
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE, val_batch_size=128)
trim = training.trimmer(manifest)   # trim(images) cuts the halo off a model's output

## 1. Label every validation patch

The manifest records how each source was degraded. We use it to split the scores later:
by **bitrate** (how hard the codec squeezed it) and by **content** (video, or an image
that slides, zooms or stands still).

In [ ]:
def bitrate_group(bpp):
    """Which bitrate group a source is in. bpp = bits per pixel per frame, None = no codec."""
    if bpp is None:
        return "no codec"
    if bpp >= 0.1:
        return "bpp 0.1-0.46"
    if bpp >= 0.03:
        return "bpp 0.03-0.1"
    return "bpp < 0.03"


def content_group(source):
    """A video, or an image and how the window over it moved (still, slide or zoom)."""
    if "motion" in source:
        return f"image, {source['motion']['kind']}"
    return "video"


labels = {}   # validation patch index -> its two groups
for source in manifest["splits"]["val"]["sources"]:
    first, end = source["patches"]   # this source's patches are first, first + 1, ..., end - 1
    for index in range(first, end):
        labels[index] = {"bitrate": bitrate_group(source["degradation"]["bpp"]),
                         "content": content_group(source)}

## 2. Model

`models.ReparamUpscaler` (see `helpers/models.py`). It must start exactly at bicubic.

In [ ]:
models.check_skip_is_bicubic(SCALE)
model = models.ReparamUpscaler(WIDTHS, STEM, EXPAND, scale=SCALE).to(DEVICE)
assert model.halo <= manifest["halo"], "the model needs more context than the patches have"

# learned parameters only: both models also hold the 900 frozen weights of the bicubic skip
while_training = models.count_parameters(model, learned_only=True)
after_merging = models.count_parameters(models.PlainUpscaler(WIDTHS, STEM), learned_only=True)
print(f"halo {model.halo}; {while_training:,} learned parameters while training, {after_merging:,} after merging")

bicubic = models.Resize(SCALE, "bicubic").to(DEVICE)
baseline = training.evaluate(bicubic, val_loader, trim)
start = training.evaluate(model, val_loader, trim)
metrics.print_table({"bicubic": baseline, "untrained": start})
assert abs(start["wpsnr"] - baseline["wpsnr"]) < 1e-3, "the model should start as bicubic"

## 3. Train

The recipe from notebook 4: Adam at 1e-3, cosine schedule, 100 epochs, batch 32, the
brightness-weighted L1 loss, best epoch by `wpsnr`.

The best epoch is picked on the validation patches, and the same patches score it below,
so that score is slightly optimistic. If the best epoch is close to the last one in the
plot, the difference is small.

In [ ]:
torch.manual_seed(SEED)
history, best = training.train(model, train_loader, val_loader, trim, EPOCHS,
                               learning_rate=LEARNING_RATE, loss_fn=metrics.luma_chroma_l1)
training.plot_history(history, baseline, loss_name="brightness-weighted L1")

## 4. Results

Against bilinear, bicubic and Lanczos-3 on the same validation patches. Lanczos-3 is
Pillow's: ONNX cannot express it, so it shows the best a plain filter can do.

A gain only in the "no codec" group means the model learned to sharpen. A gain at low
bitrates means it learned to clean up the codec.

In [ ]:
class Lanczos3(torch.nn.Module):
    """Pillow's Lanczos-3 as a module (per patch, on the CPU) so it can be scored."""

    def forward(self, x):
        size = (x.shape[3] * SCALE, x.shape[2] * SCALE)   # Pillow wants (width, height)
        images = (x.clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy()
        upscaled = [np.asarray(Image.fromarray(image).resize(size, Image.LANCZOS)) for image in images]
        return torch.from_numpy(np.stack(upscaled)).permute(0, 3, 1, 2).float().div(255).to(x.device)


methods = {
    "bilinear": models.Resize(SCALE, "bilinear").to(DEVICE),
    "bicubic": bicubic,
    "lanczos3": Lanczos3(),
    NAME: model,
}
results = {}
for name, method in methods.items():
    results[name] = training.evaluate(method, val_loader, trim)


def gain_over_bicubic(name, scores):
    """The last column of the table."""
    return f"{scores['wpsnr'] - results['bicubic']['wpsnr']:+8.2f} vs bicubic"


metrics.print_table(results, extra=gain_over_bicubic)

In [ ]:
"""wpsnr of every method within each group of validation patches."""


def as_column(value, signed=False):
    """One number of the table, or "-" for an empty group."""
    if value is None:
        return f"{'-':>15}"
    return f"{value:+15.2f}" if signed else f"{value:15.2f}"


def print_by_group(key, groups):
    """Print the table for one way of grouping the patches ("bitrate" or "content")."""
    # which validation patches are in each group
    members = {}
    for group in groups:
        members[group] = [index for index, label in labels.items() if label[key] == group]

    # score every method on each group (None for a group with no patches)
    scores = {}
    for name, method in methods.items():
        scores[name] = []
        for group in groups:
            if not members[group]:
                scores[name].append(None)
                continue
            loader = DataLoader(Subset(val_loader.dataset, members[group]), batch_size=128)
            scores[name].append(training.evaluate(method, loader, trim)["wpsnr"])

    gains = []
    for model_score, bicubic_score in zip(scores[NAME], scores["bicubic"]):
        gains.append(None if model_score is None else model_score - bicubic_score)

    print(f"\nwpsnr (dB) by {key}")
    print(f"{'':<16}" + "".join(f"{group:>15}" for group in groups))
    print(f"{'patches':<16}" + "".join(f"{len(members[group]):>15}" for group in groups))
    for name in methods:
        print(f"{name:<16}" + "".join(as_column(value) for value in scores[name]))
    print(f"{'gain':<16}" + "".join(as_column(value, signed=True) for value in gains))


print_by_group("bitrate", ["no codec", "bpp 0.1-0.46", "bpp 0.03-0.1", "bpp < 0.03"])
print_by_group("content", ["image, still", "image, slide", "image, zoom", "video"])

In [ ]:
"""Bicubic, the model and the target, with the brightness error of each."""

val_set = val_loader.dataset
picks = [i for i in (0, 400, 900, 1600) if i < len(val_set)]
lr = torch.stack([val_set[i][0] for i in picks]).to(DEVICE)
hr = torch.stack([val_set[i][1] for i in picks]).to(DEVICE)
with torch.no_grad():
    outputs = {"bicubic": trim(models.bicubic(lr, SCALE)).clamp(0, 1),
               NAME: trim(model.eval()(lr)).clamp(0, 1)}

# The error picture is the brightness (Y) difference, in grey. Errors are small, so it is
# scaled up: an error of 0.15 or more shows as white.
target_y = metrics.rgb_to_yuv(hr)[:, :1]
errors = {}
for name, out in outputs.items():
    error_y = (metrics.rgb_to_yuv(out)[:, :1] - target_y).abs()
    errors[f"error {name}"] = (error_y / 0.15).repeat(1, 3, 1, 1)   # grey = the same in R, G and B

training.show_patches({**outputs, "target": hr, **errors})

## 5. Merge the branches

The three branches of each layer are added into one convolution. The merged model must
give the same output and the same score - it is the one that ships.

In [ ]:
shipped = model.eval().collapsed()
training.assert_same_model(model, shipped)
merged_scores = training.evaluate(shipped, val_loader, trim)
assert abs(merged_scores["wpsnr"] - results[NAME]["wpsnr"]) < 0.01, "merging lost quality"
print(f"merged: {models.count_parameters(shipped, learned_only=True):,} learned parameters, "
      f"wpsnr {merged_scores['wpsnr']:.2f} dB")

## 6. Save and reload

In [ ]:
training.save_checkpoint(CHECKPOINT, shipped, architecture="PlainUpscaler",
                         arguments={"widths": list(WIDTHS), "stem": STEM},
                         trained_as=f"ReparamUpscaler(expand={EXPAND}), merged", scale=SCALE,
                         halo=shipped.halo, epochs=EPOCHS, best_epoch=best["epoch"],
                         val=merged_scores, val_bicubic=results["bicubic"],
                         degradation=manifest["degradation"])

# rebuild the model from the file alone, the way a later notebook would
checkpoint = training.load_checkpoint(CHECKPOINT)
arguments = checkpoint["arguments"]
restored = models.PlainUpscaler(tuple(arguments["widths"]), arguments["stem"], scale=checkpoint["scale"]).to(DEVICE)
restored.load_state_dict(checkpoint["state_dict"])
training.assert_same_model(shipped, restored)
print("reloaded model is identical")

## 7. Publish

Into `dist/upscale/`, at the tile a client runs: a **320x180 LR step** (328x188 input at
halo 4, 640x360 kept). Published:

- the model at **float32** and **float16**, and the float16 one as a **whole 1080p frame**;
- **bilinear** and **bicubic** as `Resize` on the same tile and frame, as baselines.

Every graph is checked against torch (on noise and on a real picture) before timing.

In [ ]:
"""A real held-out picture, sent through the stream at 0.1 bits per pixel, and the
checks every published graph goes through."""

# the first DIV2K test picture that is at least 1920x1080, cropped to exactly that
for path in sources.files("div2k", "test"):
    image = sources.load_image(path)
    if image.shape[0] >= FRAME_HEIGHT and image.shape[1] >= FRAME_WIDTH:
        real = image[:FRAME_HEIGHT, :FRAME_WIDTH]
        break
_, real_lr = degrade.degrade(real, degrade.Degradation(scale=SCALE, bpp=0.1))


def real_tile(height, width):
    """The top-left height x width of the real LR picture, as a float (1, 3, h, w) array.

    Where the picture is smaller, its edge pixels are repeated to fill the shape."""
    crop = real_lr[:height, :width].transpose(2, 0, 1)[None].astype(np.float32) / 255
    missing_h, missing_w = height - crop.shape[2], width - crop.shape[3]
    return np.pad(crop, ((0, 0), (0, 0), (0, missing_h), (0, missing_w)), mode="edge")


def input_shape(step_height, step_width, halo):
    """The model input for a step: the step plus the halo on every side, as (height, width)."""
    return (step_height + 2 * halo, step_width + 2 * halo)


def check_and_time(info, module, shape, dtype, tolerance):
    """Check a written graph against the torch module (on noise and on the real picture),
    then time it on the CPU, store the time in the file and print one line about it."""
    noise = np.random.rand(1, 3, *shape).astype(np.float32)
    for x in (noise, real_tile(*shape)):
        webexport.check_graph(info["path"], module, x, dtype, tolerance)
    ms = webexport.time_graph(info["path"], shape, dtype)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms))

In [ ]:
"""The model: float32 and float16 tiles, and the float16 one as a whole frame."""

tile_name = f"tile{STEP_WIDTH}x{STEP_HEIGHT}"
label = f"{NAME} (stream-degraded)"
halo = restored.halo
tile_shape = input_shape(STEP_HEIGHT, STEP_WIDTH, halo)
frame_shape = input_shape(FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, halo)

fp32 = webexport.export(restored, f"upscale_{NAME}_{tile_name}.onnx", tile_shape,
                        label=f"{label} fp32", halo=halo)
check_and_time(fp32, restored, tile_shape, np.float32, tolerance=2e-3)

fp16 = webexport.to_float16(fp32, f"upscale_{NAME}_fp16_{tile_name}.onnx")
webexport.write_metadata(fp16["path"], {"label": f"{label} fp16"})
check_and_time(fp16, restored, tile_shape, np.float16, tolerance=4e-3)

frame = webexport.to_frame(fp16, f"upscale_{NAME}_fp16_{tile_name}_frame.onnx", FRAME_WIDTH, FRAME_HEIGHT)
check_and_time(frame, restored, frame_shape, np.float16, tolerance=4e-3)

In [ ]:
"""The baselines: bilinear and bicubic as one `Resize`, on the same tile and frame.

float32 only: the float16 converter breaks `Resize`, and a filter has no weights to shrink."""

for mode, halo in (("bilinear", 1), ("bicubic", 2)):   # how far each filter reads
    module = models.Resize(SCALE, mode)
    tile_shape = input_shape(STEP_HEIGHT, STEP_WIDTH, halo)
    frame_shape = input_shape(FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, halo)

    tile = webexport.export(module, f"upscale_static_{mode}_{tile_name}.onnx", tile_shape,
                            label=f"static {mode} (Resize) fp32", halo=halo)
    check_and_time(tile, module, tile_shape, np.float32, tolerance=2e-3)

    frame = webexport.to_frame(tile, f"upscale_static_{mode}_{tile_name}_frame.onnx", FRAME_WIDTH, FRAME_HEIGHT)
    check_and_time(frame, module, frame_shape, np.float32, tolerance=2e-3)

print(f"\nwritten to {webexport.MODELS_DIR}")

## 8. One whole frame

A held-out picture sent through the stream at two bitrates (the host's default, and a
tenth of it), upscaled whole by the reloaded model. The frame edge is padded by
repeating edge pixels, like a tiled client does.

In [ ]:
crop = (slice(300, 556), slice(700, 1156))   # the part of the frame shown, in HR pixels
halo = restored.halo
fig, axes = plt.subplots(2, 3, figsize=(14, 6.4))
for row, bpp in enumerate((degrade.STREAM_BPP, degrade.STREAM_BPP / 10)):
    # a still screen after one second: the kind of frame a stream shows most
    clip = np.repeat(real[None], 30, axis=0)
    truth, lr = degrade.degrade(clip, degrade.Degradation(scale=SCALE, bpp=bpp))
    truth = training.to_tensor(truth[-1])
    lr = training.to_tensor(lr[-1])

    with torch.no_grad():
        # repeat the edge pixels on the left, right, top and bottom, like a tiled client
        padded = F.pad(lr, (halo, halo, halo, halo), mode="replicate")
        outputs = {"bicubic": models.bicubic(lr, SCALE).clamp(0, 1),
                   NAME: patches.trim(restored(padded), halo * SCALE).clamp(0, 1)}

    scores = {}
    for name, out in outputs.items():
        scores[f"{name}, bpp {bpp:.3f}"] = metrics.report(out, truth)
    metrics.print_table(scores)

    pictures = {**outputs, "target": truth}
    for column, (name, image) in enumerate(pictures.items()):
        axes[row][column].imshow(training.to_image(image)[crop], interpolation="nearest")
        axes[row][column].set_title(f"{name}, bpp {bpp:.3f}", fontsize=9)
        axes[row][column].axis("off")
    Image.fromarray(training.to_image(outputs[NAME])).save(training.SAMPLE_DIR / f"upscaled_reparam_bpp{bpp:.3f}.png")
fig.tight_layout()
plt.show()

## What to look at

- **The gain over bicubic by bitrate**: does it hold at low bitrates (learned the codec),
  or only without a codec (learned to sharpen)?
- **Noise**: train again with another `SEED` before trusting differences of a few tenths of a dB.
- **Tile size**: these graphs use the 320x180 step. Notebooks 1-5 publish at step 128 when
  you run them, so compare models at the same step on the benchmark page.